# 03 — Build Embedding Index (FAISS)

Build the production dense-retrieval index.

The indexing implementation lives in `src/rag/vector_store/faiss.py` and reads
the processed comments parquet incrementally so the full embedding corpus is
never materialized in RAM.

In [ ]:
from pathlib import Path
import sys
import time

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.rag.config import load_config
from src.rag.embedding.factory import EmbeddingFactory
from src.rag.preprocessing.processor import TextProcessor
from src.rag.vector_store.faiss import FAISSVectorStore
from src.rag.retrieval.embedding import EmbeddingRetriever

In [ ]:
config = load_config(PROJECT_ROOT / "configs" / "rag.yaml")

COMMENTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "comments_clean.parquet"
)

INDEX_DIR = (
    PROJECT_ROOT
    / "data"
    / "indexes"
    / "product_comments_embedding"
)

CHUNK_SIZE = 5_000
ENCODE_BATCH_SIZE = 64
OVERWRITE = True

## Build index

In [ ]:
processor = TextProcessor()

embedding_model = EmbeddingFactory.create(
    provider=config["embedding"]["provider"],
    model_name=config["embedding"]["model"],
)

start = time.perf_counter()

manifest = FAISSVectorStore.build_from_parquet(
    input_path=COMMENTS_PATH,
    output_path=INDEX_DIR,
    embedding_model=embedding_model,
    processor=processor,
    chunk_size=CHUNK_SIZE,
    encode_batch_size=ENCODE_BATCH_SIZE,
    overwrite=OVERWRITE,
)

print("Elapsed minutes:", round((time.perf_counter() - start) / 60, 2))
print(manifest)

## Smoke test

In [ ]:
vector_store = FAISSVectorStore().load(INDEX_DIR)

retriever = EmbeddingRetriever(
    embedding_model=embedding_model,
    vector_store=vector_store,
    processor=processor,
)

query = "ضد آفتاب مناسب پوست چرب"

start = time.perf_counter()
results = retriever.retrieve(query, top_k=5)
latency_ms = (time.perf_counter() - start) * 1000

print("Latency (ms):", round(latency_ms, 2))
display(results[["id", "product_id", "score", "body"]])